In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-chroma langchain-google-genai chromadb pypdf sentence-transformers gradio

In [ ]:
import os
import warnings
import gradio as gr
from getpass import getpass

# Suppress minor SDK warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. API Key Setup
api_key = os.environ.get("GEMINI_KEY")
if not api_key:
    api_key = getpass("Enter your Google Gemini API Key: ")
    os.environ["GEMINI_KEY"] = api_key

# 2. Initialize LLM with the supported gemini-3.6-flash model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key
)

# 3. Helper to handle String vs List outputs from LangChain/Gemini SDK
def extract_text(response):
    """Safely extracts plain string content from LLM response object or dictionary."""
    content = response.content if hasattr(response, 'content') else response
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        text_parts = [
            item.get('text', '') if isinstance(item, dict) else str(item)
            for item in content
        ]
        return "\n".join(text_parts)
    return str(content)

# 4. Global State & Embedding Model Initialization
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = None
retriever = None
mastery_log = []

# 5. Ingestion Handler (Supports both UI uploads & local Colab file paths)
def process_document(pdf_file):
    global vectorstore, retriever
    if pdf_file is None:
        return "⚠️ Please upload a PDF file first."
    
    file_path = pdf_file.name if hasattr(pdf_file, 'name') else pdf_file
    
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        splits = text_splitter.split_documents(docs)
        
        vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        return f"✅ Indexed successfully! ({len(splits)} text chunks prepared)"
    except Exception as e:
        return f"❌ Error loading PDF: {str(e)}"

# 6. Smart Study Buddy Feature Functions
def explain_concept(concept, style):
    if not concept or not concept.strip():
        return "⚠️ Please enter a concept name."
    if retriever is None:
        return "⚠️ Please upload and index a PDF document first!"
    
    try:
        docs = retriever.invoke(concept)
        context = "\n\n".join([d.page_content for d in docs])
        
        prompt = f"""You are Smart Study Buddy 🎓, an AI tutor. 
Explain '{concept}' based strictly on this study material:

Context:
{context}

Style: {style}
Provide a clear, engaging explanation with key bullet points.
"""
        res = llm.invoke(prompt)
        return extract_text(res)
    except Exception as e:
        return f"❌ Execution Error: {str(e)}"

def generate_quiz(topic, num_questions):
    if not topic or not topic.strip():
        return "⚠️ Please enter a topic."
    if retriever is None:
        return "⚠️ Please upload and index a PDF document first!"
    
    try:
        docs = retriever.invoke(topic)
        context = "\n\n".join([d.page_content for d in docs])
        
        prompt = f"""Generate a {num_questions}-question practice quiz for '{topic}' using this material:

Context:
{context}

Format:
- Multiple choice (A, B, C, D)
- Answer Key with explanations at the bottom.
"""
        res = llm.invoke(prompt)
        return extract_text(res)
    except Exception as e:
        return f"❌ Execution Error: {str(e)}"

def track_progress(topic, score_status):
    global mastery_log
    if not topic:
        return "⚠️ Please enter a topic name."
    status_icon = "✅ Mastered" if score_status == "Passed" else "⚠️ Needs Review"
    mastery_log.append({"Topic": topic, "Status": status_icon})
    
    report = "### 📊 Mastery Dashboard\n"
    for idx, entry in enumerate(mastery_log, 1):
        report += f"{idx}. **{entry['Topic']}** — {entry['Status']}\n"
    return report

def generate_schedule(available_hours, target_days):
    try:
        prompt = f"""Create a {target_days}-day study schedule for someone with {available_hours} hours available per day.
Format as a clean Markdown table with columns: Day, Focus Area, Task, Break Time.
"""
        res = llm.invoke(prompt)
        return extract_text(res)
    except Exception as e:
        return f"❌ Execution Error: {str(e)}"

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Smart Study Buddy 🎓") as app:
    gr.Markdown("# 🎓 Smart Study Buddy — AI RAG Tutor")
    gr.Markdown("Upload your study materials (PDF) and let your AI tutor help you master concepts, generate quizzes, track progress, and build study plans.")
    
    with gr.Row():
        pdf_input = gr.File(label="Upload PDF Document", file_types=[".pdf"])
        upload_btn = gr.Button("⚡ Index Document", variant="primary")
    
    upload_status = gr.Textbox(label="Status", interactive=False)
    upload_btn.click(process_document, inputs=[pdf_input], outputs=[upload_status])
    
    with gr.Tabs():
        with gr.TabItem("💡 Concept Explainer"):
            concept_input = gr.Textbox(label="Concept to Explain", placeholder="e.g., What is RAG?")
            style_input = gr.Radio(["Simple Metaphor", "Academic Breakdown", "Real-world Example"], value="Simple Metaphor", label="Style")
            explain_btn = gr.Button("Explain Concept", variant="primary")
            explain_output = gr.Markdown()
            explain_btn.click(explain_concept, inputs=[concept_input, style_input], outputs=[explain_output])

        with gr.TabItem("📝 Practice Quiz Generator"):
            quiz_topic = gr.Textbox(label="Quiz Topic", placeholder="e.g., Chapter 1 Basics")
            num_q = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Number of Questions")
            quiz_btn = gr.Button("Generate Quiz", variant="primary")
            quiz_output = gr.Markdown()
            quiz_btn.click(generate_quiz, inputs=[quiz_topic, num_q], outputs=[quiz_output])

        with gr.TabItem("📈 Progress Tracker"):
            tracker_topic = gr.Textbox(label="Topic Name")
            status_input = gr.Radio(["Passed", "Needs Work"], label="Status", value="Passed")
            track_btn = gr.Button("Log Topic")
            tracker_output = gr.Markdown()
            track_btn.click(track_progress, inputs=[tracker_topic, status_input], outputs=[tracker_output])

        with gr.TabItem("📅 Study Planner"):
            hours = gr.Number(label="Daily Study Hours", value=2)
            days = gr.Number(label="Days to Exam", value=7)
            schedule_btn = gr.Button("Create Plan", variant="primary")
            schedule_output = gr.Markdown()
            schedule_btn.click(generate_schedule, inputs=[hours, days], outputs=[schedule_output])

# Launch app with appropriate concurrency settings
app.queue(default_concurrency_limit=5).launch(debug=True, share=True)

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Smart Study Buddy 🎓") as app:
    gr.Markdown("# 🎓 Smart Study Buddy — AI RAG Learning Assistant")
    gr.Markdown("Upload your study material (PDF) and let your AI tutor help you master concepts, test your knowledge, and plan your study routine.")

    with gr.Row():
        pdf_input = gr.File(label="Upload Textbook / Study Material (PDF)", file_types=[".pdf"])
        upload_btn = gr.Button("⚡ Index Document", variant="primary")

    upload_status = gr.Textbox(label="System Status", interactive=False)
    upload_btn.click(process_document, inputs=[pdf_input], outputs=[upload_status])

    with gr.Tabs():
        # Tab 1: Concept Explainer
        with gr.TabItem("💡 Concept Explainer"):
            concept_input = gr.Textbox(label="What concept do you want explained?", placeholder="e.g., Photosynthesis, Neural Networks, Supply and Demand")
            style_input = gr.Radio(
                ["Simple 5-year-old metaphor", "Detailed academic breakdown", "Practical real-world example"],
                label="Explanation Style",
                value="Simple 5-year-old metaphor"
            )
            explain_btn = gr.Button("Explain Concept", variant="primary")
            explain_output = gr.Markdown(label="Explanation Output")
            explain_btn.click(explain_concept, inputs=[concept_input, style_input], outputs=[explain_output])

        # Tab 2: Quiz Generator
        with gr.TabItem("📝 Practice Quiz Generator"):
            quiz_topic = gr.Textbox(label="Quiz Topic", placeholder="e.g., Chapter 3, Mitosis, Key Terms")
            num_q = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Number of Questions")
            quiz_btn = gr.Button("Generate Practice Quiz", variant="primary")
            quiz_output = gr.Markdown(label="Quiz")
            quiz_btn.click(generate_quiz, inputs=[quiz_topic, num_q], outputs=[quiz_output])

        # Tab 3: Mastery Tracker
        with gr.TabItem("📈 Progress & Mastery Tracker"):
            tracker_topic = gr.Textbox(label="Topic Name")
            status_input = gr.Radio(["Passed", "Needs Work"], label="Assessment Status", value="Passed")
            track_btn = gr.Button("Log Progress")
            tracker_output = gr.Markdown(label="Dashboard")
            track_btn.click(track_progress, inputs=[tracker_topic, status_input], outputs=[tracker_output])

        # Tab 4: Study Planner
        with gr.TabItem("📅 Personalized Study Schedule"):
            hours = gr.Number(label="Available Study Hours Per Day", value=2)
            days = gr.Number(label="Number of Days Until Exam/Target", value=7)
            schedule_btn = gr.Button("Generate Schedule", variant="primary")
            schedule_output = gr.Markdown(label="Study Schedule")
            schedule_btn.click(generate_schedule, inputs=[hours, days], outputs=[schedule_output])

# Launch the app in Colab
app.launch(debug=True, share=True)

/tmp/ipykernel_718/2609738247.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Smart Study Buddy 🎓") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e59eae73f36b74288a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py", line 4177, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        **request,
        ^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 6270, in generate_content
    response = self._generate_content(
        model=model, contents=contents, config=parsed_config_to_call
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 4707, in _generate_content
    response = self._api_client.request(
        'post', path, request_dict, http_options
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/_api_client.py", line 1750, in request
    response = self._request(http_request, http_options, stream=False)
  File "/usr/local/lib/python3.13/dist-packages/google/gena